In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/adamgetzkin/test-ground-truth-csv/test_ground_truth.csv
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/target_pairs.csv
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/train_labels.csv
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/train.csv
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/test.csv
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/lagged_test_labels/test_labels_lag_1.csv
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/lagged_test_labels/test_labels_lag_4.csv
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/lagged_test_labels/test_labels_lag_3.csv
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/lagged_test_labels/test_labels_lag_2.csv
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/kaggle_evaluation/mitsui_inference_server.py
/kaggle/input/competitions/mitsui-commodity-prediction-challenge/kaggle_ev

## Imports & Config

In [2]:
import os
import warnings
import pandas as pd
import numpy as np
import lightgbm as lgb
import polars as pl
import kaggle_evaluation.mitsui_inference_server

warnings.filterwarnings('ignore')

DATA_DIR = '/kaggle/input/competitions/mitsui-commodity-prediction-challenge/'


## Predict Function

Key improvement over v5: **pair-specific features** from `target_pairs.csv`.
Each target measures the relationship between two specific assets, so we build
relative return, spread, ratio, mean-reversion, and rolling correlation features
for each pair directly.


In [3]:
# Global state
_models       = {}
_feature_cols = {}
_target_cols  = []

def _to_pd(x):
    return x.to_pandas() if hasattr(x, 'to_pandas') else x

def _build_base_features(price_df, base_cols):
    """Market-wide return, momentum and cross-sectional features."""
    price_df = price_df.copy().ffill().bfill()
    feat = pd.DataFrame({'date_id': price_df['date_id'].values})
    ret1_cols, ret5_cols, ret20_cols = [], [], []
    for col in base_cols:
        prices = price_df[col]
        r1, r5, r20 = f'{col}_ret1', f'{col}_ret5', f'{col}_ret20'
        feat[r1]  = prices.pct_change(1)
        feat[r5]  = prices.pct_change(5)
        feat[r20] = prices.pct_change(20)
        feat[f'{col}_vol5']  = prices.pct_change(1).rolling(5).std()
        feat[f'{col}_mom20'] = prices.pct_change(1).rolling(20).mean()
        ret1_cols.append(r1); ret5_cols.append(r5); ret20_cols.append(r20)
    cs_mean_1  = feat[ret1_cols].mean(axis=1)
    cs_std_1   = feat[ret1_cols].std(axis=1)  + 1e-8
    cs_mean_5  = feat[ret5_cols].mean(axis=1)
    cs_std_5   = feat[ret5_cols].std(axis=1)  + 1e-8
    cs_mean_20 = feat[ret20_cols].mean(axis=1)
    cs_std_20  = feat[ret20_cols].std(axis=1) + 1e-8
    rank1_df = feat[ret1_cols].rank(axis=1, pct=True)
    rank5_df = feat[ret5_cols].rank(axis=1, pct=True)
    for col in base_cols:
        feat[f'{col}_ret1_cs_z']      = (feat[f'{col}_ret1']  - cs_mean_1)  / cs_std_1
        feat[f'{col}_ret5_cs_z']      = (feat[f'{col}_ret5']  - cs_mean_5)  / cs_std_5
        feat[f'{col}_ret20_cs_z']     = (feat[f'{col}_ret20'] - cs_mean_20) / cs_std_20
        feat[f'{col}_ret1_cs_rank']   = rank1_df[f'{col}_ret1']
        feat[f'{col}_ret5_cs_rank']   = rank5_df[f'{col}_ret5']
        feat[f'{col}_ret1_cs_rank_sq']= rank1_df[f'{col}_ret1'] ** 2
    feat = feat.replace([np.inf, -np.inf], np.nan)
    return feat

def _build_pair_features(price_df, asset_a, asset_b):
    """
    Build features specific to a target pair (asset_a, asset_b).
    These directly capture what the target measures.
    """
    price_df = price_df.copy().ffill().bfill()
    feat = pd.DataFrame({'date_id': price_df['date_id'].values})
    if asset_a not in price_df.columns or asset_b not in price_df.columns:
        return feat
    pa = price_df[asset_a]
    pb = price_df[asset_b]
    ra1  = pa.pct_change(1);  rb1  = pb.pct_change(1)
    ra5  = pa.pct_change(5);  rb5  = pb.pct_change(5)
    ra20 = pa.pct_change(20); rb20 = pb.pct_change(20)
    # Relative returns (direct signal for pair target)
    feat['pair_rel_ret1']  = ra1  - rb1
    feat['pair_rel_ret5']  = ra5  - rb5
    feat['pair_rel_ret20'] = ra20 - rb20
    # Price spread and ratio
    feat['pair_spread']    = pa - pb
    feat['pair_ratio']     = pa / (pb + 1e-8)
    feat['pair_ratio_ma5'] = feat['pair_ratio'].rolling(5).mean()
    feat['pair_ratio_ma20']= feat['pair_ratio'].rolling(20).mean()
    # Mean reversion signal: spread z-score
    spread_ma20 = feat['pair_spread'].rolling(20).mean()
    spread_std20 = feat['pair_spread'].rolling(20).std() + 1e-8
    feat['pair_spread_z20'] = (feat['pair_spread'] - spread_ma20) / spread_std20
    # Relative volatility
    feat['pair_rel_vol5']  = ra1.rolling(5).std() - rb1.rolling(5).std()
    # Rolling correlation between the two assets
    feat['pair_corr20']    = ra1.rolling(20).corr(rb1)
    feat['pair_corr5']     = ra1.rolling(5).corr(rb1)
    feat = feat.replace([np.inf, -np.inf], np.nan)
    return feat

def predict(test, label_lags_1_batch, label_lags_2_batch, label_lags_3_batch, label_lags_4_batch):
    global _models, _feature_cols, _target_cols

    if not _models:
        print('First call — loading data and training models...')

        train  = pd.read_csv(DATA_DIR + 'train.csv').sort_values('date_id').reset_index(drop=True)
        labels = pd.read_csv(DATA_DIR + 'train_labels.csv').sort_values('date_id').reset_index(drop=True)
        pairs  = pd.read_csv(DATA_DIR + 'target_pairs.csv')

        _target_cols = [c for c in labels.columns if c != 'date_id']
        base_cols    = [c for c in train.columns  if c != 'date_id']

        # Build a lookup: target -> (asset_a, asset_b)
        pair_lookup = {}
        if len(pairs.columns) >= 3:
            target_col = pairs.columns[0]
            a_col      = pairs.columns[1]
            b_col      = pairs.columns[2]
            for _, row in pairs.iterrows():
                pair_lookup[row[target_col]] = (row[a_col], row[b_col])
        print(f'Loaded {len(pair_lookup)} target pairs')

        print('Building base features...')
        base_feat_df = _build_base_features(train, base_cols)
        base_feat_cols = [c for c in base_feat_df.columns if c != 'date_id']

        df_base = base_feat_df.merge(labels, on='date_id')

        # Add lagged targets
        for t in _target_cols:
            df_base[f'{t}_lag1'] = df_base[t].shift(2)
            df_base[f'{t}_lag2'] = df_base[t].shift(3)
            df_base[f'{t}_lag_diff'] = df_base[f'{t}_lag1'] - df_base[f'{t}_lag2']

        cutoff   = int(df_base['date_id'].quantile(0.8))
        train_df = df_base[df_base['date_id'] <= cutoff].copy()
        val_df   = df_base[df_base['date_id'] >  cutoff].copy()
        print(f'Train rows: {len(train_df)} | Val rows: {len(val_df)}')

        print('Training models (with pair features)...')
        for i, t in enumerate(_target_cols):
            if i % 20 == 0:
                print(f'Training model {i}/{len(_target_cols)}...')

            mask_train = ~train_df[t].isna()
            mask_val   = ~val_df[t].isna()
            if mask_train.sum() < 10:
                continue

            # Build pair-specific features if we know the pair
            pair_feat_cols = []
            if t in pair_lookup:
                asset_a, asset_b = pair_lookup[t]
                pair_feat = _build_pair_features(train, asset_a, asset_b)
                pair_feat_cols = [c for c in pair_feat.columns if c != 'date_id']
                train_pair = pair_feat.iloc[:len(train_df)][pair_feat_cols].values
                val_pair   = pair_feat.iloc[len(train_df):len(train_df)+len(val_df)][pair_feat_cols].values
                X_train_pair = pd.DataFrame(train_pair, index=train_df.index, columns=pair_feat_cols)
                X_val_pair   = pd.DataFrame(val_pair,   index=val_df.index,   columns=pair_feat_cols)
            
            lag_cols = [f'{t}_lag1', f'{t}_lag2', f'{t}_lag_diff']
            model_feat_cols = base_feat_cols + lag_cols + pair_feat_cols

            if t in pair_lookup:
                X_train = pd.concat([train_df.loc[mask_train, base_feat_cols + lag_cols].reset_index(drop=True),
                                     X_train_pair.loc[mask_train].reset_index(drop=True)], axis=1)
                X_val   = pd.concat([val_df.loc[mask_val, base_feat_cols + lag_cols].reset_index(drop=True),
                                     X_val_pair.loc[mask_val].reset_index(drop=True)], axis=1)
            else:
                X_train = train_df.loc[mask_train, base_feat_cols + lag_cols]
                X_val   = val_df.loc[mask_val,   base_feat_cols + lag_cols]

            _feature_cols[t] = model_feat_cols

            model = lgb.LGBMRegressor(
                objective='mae',
                n_estimators=500,
                learning_rate=0.03,
                max_depth=4,
                num_leaves=20,
                min_child_samples=100,
                subsample=0.8,
                colsample_bytree=0.7,
                reg_alpha=0.1,
                reg_lambda=5.0,
                random_state=42,
                n_jobs=-1,
                verbose=-1,
            )
            model.fit(
                X_train, train_df.loc[mask_train, t],
                eval_set=[(X_val, val_df.loc[mask_val, t])],
                callbacks=[lgb.early_stopping(40, verbose=False)]
            )
            _models[t] = model

        print(f'Training complete. {len(_models)} models ready.')
        predict._base_cols  = base_cols
        predict._train      = train[['date_id'] + base_cols].tail(150).copy()
        predict._pair_lookup= pair_lookup
        predict._full_train = train[['date_id'] + base_cols].copy()

    # --- INFERENCE ---
    test_row = _to_pd(test).copy().apply(pd.to_numeric, errors='coerce')
    lag1_df  = _to_pd(label_lags_1_batch).copy().apply(pd.to_numeric, errors='coerce')
    lag2_df  = _to_pd(label_lags_2_batch).copy().apply(pd.to_numeric, errors='coerce')

    predict._train = pd.concat(
        [predict._train, test_row[['date_id'] + predict._base_cols]],
        ignore_index=True
    ).tail(150)

    base_feat     = _build_base_features(predict._train, predict._base_cols)
    base_test_row = base_feat.iloc[[-1]].reset_index(drop=True)

    preds = {}
    for t in _target_cols:
        if t not in _models:
            preds[t] = np.nan
            continue

        l1 = lag1_df[t].values[0] if t in lag1_df.columns and not lag1_df[t].isna().all() else np.nan
        l2 = lag2_df[t].values[0] if t in lag2_df.columns and not lag2_df[t].isna().all() else np.nan

        row = base_test_row.copy()
        row[f'{t}_lag1']     = l1
        row[f'{t}_lag2']     = l2
        row[f'{t}_lag_diff'] = l1 - l2

        # Add pair features if available
        if t in predict._pair_lookup:
            asset_a, asset_b = predict._pair_lookup[t]
            pair_feat = _build_pair_features(predict._train, asset_a, asset_b)
            pair_row  = pair_feat.iloc[[-1]].reset_index(drop=True)
            pair_feat_cols = [c for c in pair_feat.columns if c != 'date_id']
            for pc in pair_feat_cols:
                row[pc] = pair_row[pc].values[0]

        feat_cols = _feature_cols.get(t, list(row.columns))
        X = row.reindex(columns=feat_cols)
        preds[t] = _models[t].predict(X)[0]

    # Cross-sectional normalization preserves rank order (fine for metric)
    pred_values = np.array(list(preds.values()), dtype=float)
    valid_mask  = ~np.isnan(pred_values)
    if valid_mask.sum() > 0:
        mean_val = np.mean(pred_values[valid_mask])
        std_val  = np.std(pred_values[valid_mask]) + 1e-8
        pred_values[valid_mask] = (pred_values[valid_mask] - mean_val) / std_val

    return pd.DataFrame([{t: v for t, v in zip(_target_cols, pred_values)}])


## Run Inference Server

In [4]:
# inference_server = kaggle_evaluation.mitsui_inference_server.MitsuiInferenceServer(predict)

# if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # inference_server.serve()
# else:
    # inference_server.run_local_gateway((DATA_DIR,))


## Generate Predictions on Test Set

In [5]:
test_df = pd.read_csv(DATA_DIR + 'test.csv')
lag1    = pd.read_csv(DATA_DIR + 'lagged_test_labels/test_labels_lag_1.csv')
lag2    = pd.read_csv(DATA_DIR + 'lagged_test_labels/test_labels_lag_2.csv')
lag3    = pd.read_csv(DATA_DIR + 'lagged_test_labels/test_labels_lag_3.csv')
lag4    = pd.read_csv(DATA_DIR + 'lagged_test_labels/test_labels_lag_4.csv')

# Create empty DataFrames with the same columns to pass if a date is missing
empty_lag1 = pd.DataFrame(columns=lag1.columns)
empty_lag2 = pd.DataFrame(columns=lag2.columns)
empty_lag3 = pd.DataFrame(columns=lag3.columns)
empty_lag4 = pd.DataFrame(columns=lag4.columns)

all_preds = []
for i in range(len(test_df)):
    if i % 20 == 0:
        print(f'Predicting row {i}/{len(test_df)}...')
        
    current_date = test_df.iloc[i]['date_id']
    
    # FIX 4: Filter lag batches strictly by date_id, not by row index!
    lag1_batch = lag1[lag1['date_id'] == current_date]
    lag2_batch = lag2[lag2['date_id'] == current_date]
    lag3_batch = lag3[lag3['date_id'] == current_date]
    lag4_batch = lag4[lag4['date_id'] == current_date]
    
    # If a date is completely missing from the lag file, pass an empty DataFrame
    l1_pass = lag1_batch if not lag1_batch.empty else empty_lag1
    l2_pass = lag2_batch if not lag2_batch.empty else empty_lag2
    l3_pass = lag3_batch if not lag3_batch.empty else empty_lag3
    l4_pass = lag4_batch if not lag4_batch.empty else empty_lag4
    
    row_pred = predict(
        pl.from_pandas(test_df.iloc[[i]]),
        pl.from_pandas(l1_pass),
        pl.from_pandas(l2_pass),
        pl.from_pandas(l3_pass),
        pl.from_pandas(l4_pass),
    )
    row_pred['date_id'] = current_date
    all_preds.append(row_pred)

pred_df = pd.concat(all_preds, ignore_index=True)
print(f'Predictions shape: {pred_df.shape}')

Predicting row 0/134...
First call — loading data and training models...
Loaded 424 target pairs
Building base features...
Train rows: 1569 | Val rows: 392
Training models (with pair features)...
Training model 0/424...
Training model 20/424...
Training model 40/424...
Training model 60/424...
Training model 80/424...
Training model 100/424...
Training model 120/424...
Training model 140/424...
Training model 160/424...
Training model 180/424...
Training model 200/424...
Training model 220/424...
Training model 240/424...
Training model 260/424...
Training model 280/424...
Training model 300/424...
Training model 320/424...
Training model 340/424...
Training model 360/424...
Training model 380/424...
Training model 400/424...
Training model 420/424...
Training complete. 424 models ready.
Predicting row 20/134...
Predicting row 40/134...
Predicting row 60/134...
Predicting row 80/134...
Predicting row 100/134...
Predicting row 120/134...
Predictions shape: (134, 425)


## Score Against Ground Truth

In [6]:
GROUND_TRUTH_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if 'ground_truth' in f.lower() or ('truth' in f.lower() and f.endswith('.csv')):
            GROUND_TRUTH_PATH = os.path.join(root, f)
            print('Found ground truth at:', GROUND_TRUTH_PATH)

assert GROUND_TRUTH_PATH is not None, 'Upload test_ground_truth.csv as a Kaggle dataset'

ground_truth = pd.read_csv(GROUND_TRUTH_PATH)

if _target_cols[0] in ground_truth.columns:
    gt_aligned   = ground_truth[ground_truth['date_id'].isin(pred_df['date_id'])].sort_values('date_id').reset_index(drop=True)
    pred_aligned = pred_df[pred_df['date_id'].isin(gt_aligned['date_id'])].sort_values('date_id').reset_index(drop=True)
    score_cols   = _target_cols
else:
    rename_map   = {_target_cols[i]: f'target_{i}' for i in range(len(_target_cols))}
    pred_renamed = pred_df[['date_id'] + _target_cols].rename(columns=rename_map)
    gt_aligned   = ground_truth[ground_truth['date_id'].isin(pred_renamed['date_id'])].sort_values('date_id').reset_index(drop=True)
    pred_aligned = pred_renamed[pred_renamed['date_id'].isin(gt_aligned['date_id'])].sort_values('date_id').reset_index(drop=True)
    score_cols   = [f'target_{i}' for i in range(424)]

def competition_score(pred_df, gt_df, target_cols):
    pred = pred_df[target_cols].copy()
    gt   = gt_df[target_cols].copy()
    daily_corrs = []
    
    for i in range(len(pred)):
        gt_row   = gt.iloc[i]
        pred_row = pred.iloc[i]
        
        valid = gt_row.notna()
        if valid.sum() < 2: continue
            
        gt_valid   = gt_row[valid]
        pred_valid = pred_row[valid]
        
        if gt_valid.std(ddof=0) == 0 or pred_valid.std(ddof=0) == 0: continue
            
        corr = np.corrcoef(
            pred_valid.rank(method='average'),
            gt_valid.rank(method='average')
        )[0, 1]
        daily_corrs.append(corr)
        
    daily_corrs = np.array(daily_corrs)
    sharpe = daily_corrs.mean() / daily_corrs.std(ddof=0)
    
    print(f'  Daily rank correlations: mean={daily_corrs.mean():.6f}, std={daily_corrs.std():.6f}')
    print(f'  Days evaluated: {len(daily_corrs)}')
    print(f'  Competition Score (Sharpe): {sharpe:.6f}')
    return sharpe, daily_corrs

print('Optimized LightGBM score:')
sharpe, daily = competition_score(pred_aligned, gt_aligned, score_cols)

Found ground truth at: /kaggle/input/datasets/adamgetzkin/test-ground-truth-csv/test_ground_truth.csv
Optimized LightGBM score:
  Daily rank correlations: mean=0.035028, std=0.159353
  Days evaluated: 134
  Competition Score (Sharpe): 0.219815


In [7]:
import joblib

# ==========================================
# 1. SAVE PREDICTIONS FOR ENSEMBLING (Full, unfiltered test rows)
# ==========================================
# Save the original column names
pred_df.to_csv('lgbm_attempt_6_preds_original_names.csv', index=False)

# Save the target_0, target_1 renamed version just in case your friend needs it
if 'pred_renamed' in locals():
    pred_renamed.to_csv('lgbm_attempt_6_preds_renamed.csv', index=False)
    print(f"✅ SUCCESS: Saved both original and renamed prediction CSVs (Rows: {len(pred_df)})")
else:
    print(f"✅ SUCCESS: Saved predictions to 'lgbm_attempt_6_preds_original_names.csv' (Rows: {len(pred_df)})")

# ==========================================
# 2. SAVE TRAINED MODELS FOR FUTURE USE
# ==========================================
save_data = {
    'models': _models,
    'feature_cols': _feature_cols
}
joblib.dump(save_data, 'lgbm_attempt_6_models.joblib')
print("✅ SUCCESS: All trained models saved to 'lgbm_attempt_6_models.joblib'")

✅ SUCCESS: Saved predictions to 'lgbm_attempt_6_preds_original_names.csv' (Rows: 134)
✅ SUCCESS: All trained models saved to 'lgbm_attempt_6_models.joblib'
